# Track A, steps A5 to A8: verifier, ablation, graph, error gallery, scorer
Run after A1 to A4. A GPU is only needed to compute embeddings (T4 is fine); the rest is CPU. Status: the evaluation code ran on synthetic images only; on synthetic data every number is about 1.0 and proves nothing. Only the hard-negative column on **real photos** counts.

In [ ]:
# 0) setup
GITHUB_USER = "YOUR_GITHUB_USERNAME"
import os, sys, json, random, numpy as np, pandas as pd
if not os.path.exists("/content/returnlens"):
    !git clone https://github.com/{GITHUB_USER}/returnlens.git /content/returnlens
%cd /content/returnlens
!git checkout track-a && git pull
sys.path.insert(0, "/content/returnlens/src")
from google.colab import drive; drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/returnlens_results"
!cd / && unzip -q -o "{OUT}/img_pool.zip"            # restores /content/img_amazon etc. (paths in pool.json point there)
POOL = json.load(open(f"{OUT}/pool.json")); SPLITS = json.load(open(f"{OUT}/splits.json"))
random.seed(0); np.random.seed(0)
print(len(POOL), "images;", {k: len(v) for k, v in SPLITS.items()}, "products per split")
!pip -q install transformers faiss-cpu opencv-python-headless imagehash lightgbm networkx scikit-learn joblib
import pool, retrieval, evaluate as ev
from build_pairs import build_pairs
from graph_verifier import fit_verifier

## Rebuild pairs with the embedding feature `emb_cos`
Same seed as A1, so the pairs are the same ones, now with a fine-tuned cosine column. Takes a few minutes.

In [ ]:
CKPT = f"{OUT}/embedder.pt"
emb_fn = retrieval.finetuned(CKPT)
df = build_pairs(POOL, n_variants=3, emb_fn=emb_fn)
splits = {k: set(SPLITS[k]) for k in SPLITS}
tr, va, te = (df[df.group.astype(str).isin(splits[k])].reset_index(drop=True) for k in ("train", "val", "test"))
print(len(tr), len(va), len(te)); df.to_csv(f"{OUT}/pairs_with_emb.csv", index=False)

## A5. Calibrated verifier: threshold on val, report on test
Brier score and ECE say whether a predicted 0.8 really means about 80%.

In [ ]:
from baselines import pick_threshold, evaluate
clf = fit_verifier(tr)
va["p"], te["p"] = ev.score_split(clf, va), ev.score_split(clf, te)
thr = pick_threshold(va, "p")              # locked on VAL, never retuned on test
print("threshold", thr, evaluate(te, "p", thr))
from sklearn.metrics import brier_score_loss, roc_auc_score
print("AUC", round(roc_auc_score(te.y, te.p), 3), "Brier", round(brier_score_loss(te.y, te.p), 4), "ECE", round(ev.ece(te.y, te.p), 4))
rel = ev.reliability_table(te.y, te.p); display(rel); rel.to_csv(f"{OUT}/reliability.csv", index=False)

## E3. Ablation: remove one feature group at a time

In [ ]:
ab = ev.ablation(tr, va, te); display(ab); ab.to_csv(f"{OUT}/ablation.csv", index=False)

## A6 / E4. Evidence graph vs pairwise (synthetic customers: say so in the report)
Chain of re-uploads by different customers; does the graph recover the whole chain where direct pairs do not?

In [ ]:
test_pool = [p for p in POOL if str(p["product_id"]) in splits["test"]]
chain = ev.chain_experiment(test_pool, clf, emb_fn=emb_fn, n_chains=30, length=4)
display(chain); chain.to_csv(f"{OUT}/chain_experiment.csv")

## A8. Error gallery
The 10 worst false positives (hard negatives scored as reuse) and 10 worst misses. **Open the image and write one line of reason per pair** in your report.

In [ ]:
eg = ev.error_gallery(te, te.p, k=10); display(eg); eg.to_csv(f"{OUT}/error_gallery.csv", index=False)
print(ev.save_montage(eg, f"{OUT}/error_gallery.jpg"), "pairs drawn")
from IPython.display import Image as I; display(I(f"{OUT}/error_gallery.jpg"))

## A7. Package the scorer for the /evidence endpoint
Gallery = every photo in the pool, with synthetic customers (every 3rd image is flagged as a catalogue image just to exercise the flag).

In [ ]:
from score_claim import EvidenceScorer, save_artifacts
gallery = [dict(image_id=p["source_id"], path=p["path"], claim_id=f"G{i}", customer_id=f"cust{i % 40}",
                product_id=str(p["product_id"]), is_catalogue=(i % 3 == 0)) for i, p in enumerate(POOL)]
scorer = EvidenceScorer(gallery, clf, emb_fn=emb_fn)
import cv2; from evidence import transform
probe = transform(cv2.imread(POOL[7]["path"]))[0]
print(scorer.score(probe, "PROBE", customer_id="new_customer"))
print(scorer.score(cv2.imread(POOL[0]["path"])[::-1].copy(), "PROBE2", customer_id="new_customer"))
save_artifacts(scorer, f"{OUT}/artifacts")      # then locally: RETURNLENS_ARTIFACTS=<folder> uvicorn api:app

Copy the CSV tables and `error_gallery.jpg` from Drive into the repo's `results/` folder, label each table real / weak / gold / synthetic, and write the Track A section.